Building a semantic search engine for real estate listings. Instead of only matching exact words like a keyword search, will use embeddings to understand the meaning of the listing descriptions to retrieve similar listings. 

Overview 
- Implement embedding-based semantic search using sentence-transformers. Build FAISS index for fast similarity search. Compare semantic vs keyword matching quality. 

Key Deliverables 
- Sentence embeddings for all listing remarks (384 or 768 dims) ✅
- FAISS index for efficient similarity search ✅
- SemanticSearcher class with query embedding + retrieval ✅ 
- Comparison study: semantic vs BM25 keyword search 
- Latency < 100ms for 10k listings 
- Relevance evaluation on 50 query-result pairs 

In [24]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

print("Success")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5475.25it/s]


Success


In [59]:
from sentence_transformers import SentenceTransformer 
import faiss 
import numpy as np 
import pandas as pd
from rank_bm25 import BM25Okapi

Note: FAISS is a library that allows us to quickly search through large collections of vectorss to find the embeddings closest to a user's query embedding.

In [2]:
class SemanticSearcher: 
    def __init__(self): 
        self.model = SentenceTransformer('all-MiniLM-L6-v2') 
        self.index = None 
        self.listings = None 
    def build_index(self, remarks_list): 
        print(f"Encoding {len(remarks_list)} listings...") 
        embeddings = self.model.encode(remarks_list) 
        # Build FAISS index 
        dim = embeddings.shape[1] 
        self.index = faiss.IndexFlatIP(dim)  # Inner product for cosine sim 
        faiss.normalize_L2(embeddings) 
        self.index.add(embeddings) 
        self.listings = remarks_list 
    def search(self, query, top_k=10): 
        query_emb = self.model.encode([query]) 
        faiss.normalize_L2(query_emb) 
        scores, indices = self.index.search(query_emb, top_k) 
        results = [(self.listings[i], scores[0][j]) for j, i in 
        enumerate(indices[0])] 
        return results 

In [12]:
df = pd.read_csv("../data/processed/cleaned_listing_sample.csv")
remarks = df['cleaned_remarks'].fillna("").tolist()

In [17]:
pd.set_option("display.max_colwidth", None)
df['cleaned_remarks'].head()

0                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       

In [15]:
searcher = SemanticSearcher()
searcher.build_index(remarks)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2666.12it/s]


Encoding 1000 listings...


In [25]:
embeddings = model.encode(remarks)

In [26]:
embeddings.shape

(1000, 384)

Each listing has a 384-dimensional embedding

In [30]:
# Saving the generated embeddings
#np.save("../data/listing_embeddings.npy", embeddings)

In [31]:
embeddings = np.load("../data/listing_embeddings.npy")

For deliverable 2, we have to make the embeddings made searchable

In [42]:
searcher = SemanticSearcher()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3602.27it/s]


In [43]:
searcher.build_index(remarks)

Encoding 1000 listings...


In [44]:
print(searcher.index.ntotal)

1000


The FAISS index now contains 1000 listing embeddings

In [45]:
print(searcher.index.d)

384


The index was built for 384-dimensional embeddings.

Deliverable 2 is completed

Deliverable 3 is where we basically implement the SemanticSearch search() method.

Just to note, this is how we are using the class: 

Cleaned MLS remarks

        |
        v

Convert remarks into embeddings

        |
        v

Store embeddings in FAISS

        |
        v

Accept a user's search query

        |
        v

Convert query into embedding

        |
        v
        
Find similar listings

Example

In [47]:
df['cleaned_remarks'].iloc[0]

'This unique property offers two homes on one lot, creating an exceptional opportunity for both owner-occupants and investors alike. The front home features 2 bedrooms and 1 bathroom, while the rear unit offers 1 bedroom and 1 bathroom. Ideal for extended family, rental income, or a live-in-one-rent-the-other setup. The owner currently lives in one unit, which makes it easier to occupy or rent it out!'

In [48]:
results = searcher.search(
    df['cleaned_remarks'].iloc[0],
    top_k=5
)

In [49]:
df['cleaned_remarks'].iloc[0]

'This unique property offers two homes on one lot, creating an exceptional opportunity for both owner-occupants and investors alike. The front home features 2 bedrooms and 1 bathroom, while the rear unit offers 1 bedroom and 1 bathroom. Ideal for extended family, rental income, or a live-in-one-rent-the-other setup. The owner currently lives in one unit, which makes it easier to occupy or rent it out!'

In [53]:
for remark, score in results:
    print("Similarity score:",score)
    print(remark)
    print('-------')

Similarity score: 1.0
This unique property offers two homes on one lot, creating an exceptional opportunity for both owner-occupants and investors alike. The front home features 2 bedrooms and 1 bathroom, while the rear unit offers 1 bedroom and 1 bathroom. Ideal for extended family, rental income, or a live-in-one-rent-the-other setup. The owner currently lives in one unit, which makes it easier to occupy or rent it out!
-------
Similarity score: 0.75226396
Investment opportunity! The main home offers three bedrooms and two bathrooms. The property also includes additional converted living areas, bringing the total to six bedrooms, four bathrooms, and three kitchen areas. Featuring three separate entrances, the layout offers flexibility for multi-generational living or potential rental income. The home is fully fenced with parking and access from both 40th Avenue (front) and Rosedale (rear). Buyer to investigate and verify permit status, bedroom and bathroom count, and square footage o

In [54]:
query = "luxury kitchen with modern appliances"

results = searcher.search(query, top_k=5)

In [55]:
results

[('Modern Luxury Meets Elevated Design in this stunning Toll Brothers home located in the desirable Westridge collection within the gated Metropolitan Heights community. Positioned on a premium homesite, this three-story residence showcases sleek contemporary architecture, panoramic views from every level, and over $400000 in premium upgrades and enhancements. Step inside to experience an open-concept floor plan featuring dramatic floating stairs with glass railing, expansive living spaces, and an abundance of natural light. The interior has been beautifully upgraded with custom tile flooring downstairs and real wood flooring throughout the home, creating a warm yet sophisticated feel. The chef\'s kitchen is a true showpiece with premium JennAir appliances including a 48" gas range, built-in refrigerator, oversized island with Quartzite Blue Tahoe countertops, modern white acrylic cabinetry, and a stylish full-height backsplash. Designed for both comfort and entertaining, the spacious 

So far:

We have implemented the SemanticSearcher class using SentenceTransformer embeddings and FAISS retrieval. User queries are converted into 384-dimensional embeddings using all-MiniLM-L6-v2, normalized, and searched against the FAISS index. The system returns the top-k most semantically similar listing remarks ranked by cosine similarity.

Deliverable 4: Designing an experiment to compare two search approaches:
- Semantic Search: SentenceTransformer embeddings + FAISS that find listings based on the meaning and context of the remarks themselves 
- BM25 Keyword Search: Traditional information retrieval algorithm that find listings based on word overlap/frequency